# MDLM: Masked Diffusion Language Model

Reference: https://github.com/kuleshov-group/mdlm

## 0. Imports

In [1]:
import json
from typing import Optional

import torch
from huggingface_hub import hf_hub_download
from safetensors.torch import load_file
from transformers import AutoTokenizer

from mdlm_model import MDLM, MDLMConfig

/home/user/prj/mdlm/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. 모델 로드
주의: 모델은 130M 파라미터로 작은 편이라 생성 퀄리티는 낮을 수 있음

In [2]:
MODEL_ID = "kuleshov-group/mdlm-owt"
MASK_TOKEN_ID = 50257

tokenizer = AutoTokenizer.from_pretrained("gpt2")

config_path = hf_hub_download(MODEL_ID, "config.json")
with open(config_path) as f:
    cfg = json.load(f)
config = MDLMConfig(**{k: cfg[k] for k in [
    "vocab_size", "model_length", "hidden_dim", "cond_dim",
    "n_blocks", "n_heads", "dropout", "time_conditioning",
]})
model = MDLM(config)

weights_path = hf_hub_download(MODEL_ID, "model.safetensors")
state_dict = load_file(weights_path)
model.load_state_dict(state_dict)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device).eval()
print(f"Model loaded on {device} (130M params)")

Model loaded on cuda (130M params)


## 2. SUBS Parameterization

MDLM의 생성 과정 잡기술. 모델 출력 logits에서 위치별로 **두 가지로 분기**

| 위치 종류 | 처리 |
|---|---|
| **Masked** (`▒`) 위치 | logits에서 mask token을 −∞로 제한 |
| **Unmasked** (이미 드러난) 위치 | 현재 토큰을 그대로 유지 |

**왜 이렇게?**

한 번 unmask된 토큰은 다시 mask로 돌아가지 않음(Absorbing Diffusion) 즉 reverse process는 `mask → token` 방향으로만 흐름(monotonic). SUBS는 이 제약을 **모델 출력 단계에서 강제**해서 sampling이 안전하게 진행되도록 보장. 추가로 **mask token을 다음 토큰으로 예측하는 것도 방지**

In [3]:
def subs_parameterization(logits: torch.Tensor, x: torch.Tensor) -> torch.Tensor:
    # mask token으로 예측되는 것을 금지
    logits[:, :, MASK_TOKEN_ID] = -1e9
    log_probs = logits - logits.logsumexp(dim=-1, keepdim=True)

    # unmasked 위치(이미 text가 된 위치)는 현재 토큰을 유지
    unmasked = (x != MASK_TOKEN_ID)
    if unmasked.any():
        log_probs[unmasked] = -1e9
        log_probs[unmasked, x[unmasked]] = 0.0

    return log_probs

## 3. DDPM Reverse Diffusion Sampler

**전체 알고리즘.** 시간 `t`를 1→0(all-mask → all-text):

1. 현재 sequence `x_t`로 모델 forward → `p(x_0 | x_t)` 예측
2. **Reverse transition** `q(x_s | x_t, x_0)`에서 `x_s` 샘플링 (s < t, 한 step 덜 noisy)
   - 각 위치가 unmask될 확률 ∝ `p(x_0) · (t − s)`
   - mask로 남을 확률 = `s`
3. **Gumbel-max trick**으로 categorical 샘플링
4. 이미 unmask된 위치는 SUBS 덕분에 변경되지 않음

**Noise schedule.** loglinear, `σ(t) = -log(1 − (1−ε)t)`. 단순 구현에서는 `move_chance ≈ t`로 근사

In [4]:
def _format(token_ids: torch.Tensor) -> str:
    """마스크 위치는 ▒, 나머지는 decode"""
    parts = []
    for tid in token_ids:
        if tid.item() == MASK_TOKEN_ID:
            parts.append("▒")
        else:
            parts.append(tokenizer.decode([tid.item()]))
    return "".join(parts)

In [5]:
@torch.no_grad()
def sample_ddpm(
    seq_len: int = 64,
    num_steps: int = 128,
    initial_ids: Optional[torch.Tensor] = None,
    mask_positions: Optional[torch.Tensor] = None,
    print_every: int = 10,
):
    """DDPM reverse diffusion with caching."""
    eps = 1e-5

    # 초기화: initial_ids가 주어지면 거기서 시작, 아니면 전체 mask
    if initial_ids is not None:
        x = initial_ids.clone().to(device)
        seq_len = x.shape[1]
    else:
        x = torch.full((1, seq_len), MASK_TOKEN_ID, dtype=torch.long, device=device)

    if mask_positions is not None:
        mask_positions = mask_positions.to(device)

    total_masks_start = (x == MASK_TOKEN_ID).sum().item()

    # 시간 스케줄: t=1 (fully masked) → t=eps (fully denoised)
    timesteps = torch.linspace(1, eps, num_steps + 1)

    print("=" * 70)
    print(f"DDPM Sampling: {total_masks_start} masked tokens, {num_steps} steps")
    print("=" * 70)
    print()
    print(f"[t=1.000] Initial state")
    print(_format(x[0]))
    print()

    p_x0_cache = None
    x_cached = None

    for i in range(num_steps):
        t = timesteps[i].item()
        s = timesteps[i + 1].item()  # next (less noisy) time

        # Cache: x가 바뀌었을 때만 forward pass
        if x_cached is None or not torch.equal(x, x_cached):
            logits = model(x)
            log_p_x0 = subs_parameterization(logits, x)
            p_x0_cache = log_p_x0.exp()
            x_cached = x.clone()

        p_x0 = p_x0_cache

        # DDPM transition: q(x_s | x_t, x0_pred)
        # 각 위치 unmask 확률 = p(x0) * (t - s), mask 유지 확률 = s
        q_xs = p_x0 * (t - s)
        q_xs[:, :, MASK_TOKEN_ID] = s

        # Gumbel-max sampling (reparameterization trick)
        gumbel_noise = -(-torch.rand_like(q_xs).clamp(min=1e-10).log()).log()
        x_new = (q_xs.log().clamp(min=-1e9) + gumbel_noise).argmax(dim=-1)

        # 이미 unmask된 토큰은 절대 변경하지 않음
        is_masked = (x[0] == MASK_TOKEN_ID)
        if mask_positions is not None:
            is_masked = is_masked & mask_positions
        x[0, is_masked] = x_new[0, is_masked]

        # 시각화
        n_remaining = (x[0] == MASK_TOKEN_ID).sum().item()
        if mask_positions is not None:
            n_remaining = ((x[0] == MASK_TOKEN_ID) & mask_positions).sum().item()

        should_print = (
            ((i + 1) % print_every == 0 and i > 30)
            or i == num_steps - 1
        )
        if should_print:
            pct = (1 - n_remaining / max(total_masks_start, 1)) * 100
            print(f"[t={s:.3f}] Step {i+1}/{num_steps} ({pct:.0f}% revealed)")
            print(_format(x[0]))
            print()

    # Final denoising: 남은 mask를 argmax로 확정
    logits = model(x)
    log_p_x0 = subs_parameterization(logits, x)
    remaining_mask = (x[0] == MASK_TOKEN_ID)
    if mask_positions is not None:
        remaining_mask = remaining_mask & mask_positions
    if remaining_mask.any():
        x[0, remaining_mask] = log_p_x0[0, remaining_mask].argmax(dim=-1)

    final_text = tokenizer.decode(x[0], skip_special_tokens=True)
    print("=" * 70)
    print("FINAL TEXT:")
    print(final_text)
    print("=" * 70)

    return final_text

## 4. Demo: Contiguous Infilling

문장의 **가운데 절반**을 통으로 비워두고 채우는 작업

아래 셀을 실행하면 step별로 `▒` 자리가 점점 단어로 채워지는 과정

Random seed로 인해 매번 다른 결과가 나올 수 있음

In [6]:
def demo_infilling_contiguous(text: str, num_steps: int = 64):
    """문장의 가운데 절반을 통으로 비우고 채우기"""
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    n = len(token_ids)

    # 가운데 절반을 mask
    start = n // 4
    end = start + n // 2
    mask_positions = torch.zeros(n, dtype=torch.bool)
    mask_positions[start:end] = True

    masked_ids = token_ids.copy()
    for i in range(start, end):
        masked_ids[i] = MASK_TOKEN_ID

    input_ids = torch.tensor([masked_ids], dtype=torch.long)

    n_masked = mask_positions.sum().item()
    print("=" * 70)
    print("  DEMO: Contiguous Infilling (통으로 비우기)")
    print(f"  Original: {tokenizer.decode(token_ids)}")
    print(f"  Masked {n_masked}/{n} tokens in the middle")
    print("=" * 70 + "\n")

    return sample_ddpm(
        initial_ids=input_ids,
        mask_positions=mask_positions,
        num_steps=num_steps,
        print_every=num_steps // 6,
    )

In [7]:
demo_infilling_contiguous(
    "The dog ran across jumped over the tall wooden fence into the garden.", # 이 중 중간 일부는 비우고 inference
    num_steps=64,
)

  DEMO: Contiguous Infilling (통으로 비우기)
  Original: The dog ran across jumped over the tall wooden fence into the garden.
  Masked 7/14 tokens in the middle

DDPM Sampling: 7 masked tokens, 64 steps

[t=1.000] Initial state
The dog ran▒▒▒▒▒▒▒ into the garden.

[t=0.375] Step 40/64 (29% revealed)
The dog ran▒▒▒▒▒ and ran into the garden.

[t=0.219] Step 50/64 (43% revealed)
The dog ran▒ the▒▒▒ and ran into the garden.

[t=0.063] Step 60/64 (71% revealed)
The dog ran across the▒▒, and ran into the garden.

[t=0.000] Step 64/64 (100% revealed)
The dog ran across the Crosspath, and ran into the garden.

FINAL TEXT:
The dog ran across the Crosspath, and ran into the garden.


'The dog ran across the Crosspath, and ran into the garden.'

## 더 시도해볼 것

- 다른 문장으로 실험해보기 (130M 모델이므로 생성 퀄리티는 낮을 수 있음)
- `num_steps`를 늘리거나 줄여보기 — step이 적으면 caching 덕에 빠르지만 quality가 떨어질 수 있음
- `start`/`end`를 바꿔서 mask 영역의 위치·크기 조절